# Итоговая домашняя работа: Trino + PostgreSQL + MySQL + Iceberg

**Автор:** Литвинов Никита Антонович, группа МИНДА 241  
**Преподаватель:** Влад Шевченко  
**Дата:** Январь 2026

---

## Содержание

1. **Уровень 1: Подключения** — подключение к Trino, проверка каталогов PostgreSQL и MySQL
2. **Уровень 2: Агрегация данных** — SQL-запросы, работа с pandas DataFrame
3. **Уровень 3: Визуализация и Iceberg** — графики, сохранение данных в Iceberg

---

## Установка зависимостей

In [ ]:
# Установка необходимых библиотек
!pip install trino pandas matplotlib seaborn plotly --quiet

In [ ]:
# Импорт библиотек
import trino
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Настройка отображения
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Цветовая палитра для графиков
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#95C623']

---

# Уровень 1: Подключения

На этом уровне мы:
1. Подключимся к серверу Trino
2. Проверим доступность каталога PostgreSQL
3. Проверим доступность каталога MySQL

## 1.1 Подключение к Trino

In [ ]:
# Параметры подключения к Trino
TRINO_HOST = 'localhost'
TRINO_PORT = 8080
TRINO_USER = 'admin'

def get_trino_connection(catalog='system'):
    """Создает подключение к Trino с указанным каталогом."""
    return trino.dbapi.connect(
        host=TRINO_HOST,
        port=TRINO_PORT,
        user=TRINO_USER,
        catalog=catalog
    )

def execute_query(query, catalog='system'):
    """Выполняет SQL-запрос и возвращает результат в виде DataFrame."""
    conn = get_trino_connection(catalog)
    cursor = conn.cursor()
    cursor.execute(query)
    columns = [desc[0] for desc in cursor.description]
    data = cursor.fetchall()
    return pd.DataFrame(data, columns=columns)

print('✅ Функции подключения к Trino созданы')

In [ ]:
# Проверка подключения - вывод списка каталогов
catalogs_df = execute_query('SHOW CATALOGS')
print('📚 Доступные каталоги в Trino:')
catalogs_df

## 1.2 Проверка каталога PostgreSQL

In [ ]:
# Список схем в PostgreSQL
pg_schemas = execute_query('SHOW SCHEMAS FROM postgresql')
print('🐘 Схемы в каталоге PostgreSQL:')
pg_schemas

In [ ]:
# Список таблиц в схеме public
pg_tables = execute_query('SHOW TABLES FROM postgresql.public')
print('📋 Таблицы в схеме postgresql.public:')
pg_tables

In [ ]:
# Просмотр структуры таблицы customers
pg_customers_desc = execute_query('DESCRIBE postgresql.public.customers')
print('👥 Структура таблицы customers:')
pg_customers_desc

In [ ]:
# Просмотр структуры таблицы orders
pg_orders_desc = execute_query('DESCRIBE postgresql.public.orders')
print('📦 Структура таблицы orders:')
pg_orders_desc

In [ ]:
# Примеры данных из PostgreSQL
print('📊 Примеры данных из таблицы orders:')
execute_query('SELECT * FROM postgresql.public.orders LIMIT 5')

## 1.3 Проверка каталога MySQL

In [ ]:
# Список схем в MySQL
mysql_schemas = execute_query('SHOW SCHEMAS FROM mysql')
print('🐬 Схемы в каталоге MySQL:')
mysql_schemas

In [ ]:
# Список таблиц в схеме sales_db
mysql_tables = execute_query('SHOW TABLES FROM mysql.sales_db')
print('📋 Таблицы в схеме mysql.sales_db:')
mysql_tables

In [ ]:
# Просмотр структуры таблицы products
mysql_products_desc = execute_query('DESCRIBE mysql.sales_db.products')
print('🏷️ Структура таблицы products:')
mysql_products_desc

In [ ]:
# Просмотр структуры таблицы sales
mysql_sales_desc = execute_query('DESCRIBE mysql.sales_db.sales')
print('💰 Структура таблицы sales:')
mysql_sales_desc

In [ ]:
# Примеры данных из MySQL
print('📊 Примеры данных из таблицы sales:')
execute_query('SELECT * FROM mysql.sales_db.sales LIMIT 5')

### ✅ Итоги Уровня 1

Успешно выполнено:
- Подключение к серверу Trino
- Проверка каталога PostgreSQL: схемы `public`, таблицы `customers` и `orders`
- Проверка каталога MySQL: схема `sales_db`, таблицы `categories`, `products`, `sales`

---

# Уровень 2: Агрегация данных

На этом уровне мы:
1. Выполним агрегирующие SQL-запросы через Trino
2. Загрузим результаты в pandas DataFrame
3. Сформируем финальный агрегированный DataFrame

## 2.1 Агрегация данных из PostgreSQL

In [ ]:
# Запрос 1: Ежемесячная статистика заказов
query_orders_monthly = '''
SELECT 
    date_trunc('month', order_date) AS month,
    COUNT(*) AS orders_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount,
    SUM(shipping_cost) AS total_shipping
FROM postgresql.public.orders
GROUP BY date_trunc('month', order_date)
ORDER BY month
'''

df_orders_monthly = execute_query(query_orders_monthly)
print('📅 Ежемесячная статистика заказов (PostgreSQL):')
df_orders_monthly

In [ ]:
# Запрос 2: Top-5 клиентов по сумме заказов
query_top_customers = '''
SELECT 
    c.customer_name,
    c.segment,
    c.city,
    COUNT(o.order_id) AS orders_count,
    SUM(o.amount) AS total_amount,
    AVG(o.amount) AS avg_order_amount
FROM postgresql.public.customers c
JOIN postgresql.public.orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_name, c.segment, c.city
ORDER BY total_amount DESC
LIMIT 5
'''

df_top_customers = execute_query(query_top_customers)
print('🏆 Top-5 клиентов по сумме заказов:')
df_top_customers

In [ ]:
# Запрос 3: Статистика по сегментам клиентов
query_segments = '''
SELECT 
    c.segment,
    COUNT(DISTINCT c.customer_id) AS customers_count,
    COUNT(o.order_id) AS orders_count,
    SUM(o.amount) AS total_revenue,
    AVG(o.amount) AS avg_order_value
FROM postgresql.public.customers c
LEFT JOIN postgresql.public.orders o ON c.customer_id = o.customer_id
GROUP BY c.segment
ORDER BY total_revenue DESC
'''

df_segments = execute_query(query_segments)
print('📊 Статистика по сегментам клиентов:')
df_segments

## 2.2 Агрегация данных из MySQL

In [ ]:
# Запрос 4: Ежемесячные продажи по категориям
query_sales_category = '''
SELECT 
    date_trunc('month', s.sale_date) AS month,
    c.category_name,
    SUM(s.quantity) AS total_quantity,
    SUM(s.quantity * s.unit_price * (1 - s.discount_percent/100)) AS revenue
FROM mysql.sales_db.sales s
JOIN mysql.sales_db.products p ON s.product_id = p.product_id
JOIN mysql.sales_db.categories c ON p.category_id = c.category_id
GROUP BY date_trunc('month', s.sale_date), c.category_name
ORDER BY month, revenue DESC
'''

df_sales_category = execute_query(query_sales_category)
print('📈 Ежемесячные продажи по категориям (MySQL):')
df_sales_category

In [ ]:
# Запрос 5: Top-10 продуктов по выручке
query_top_products = '''
SELECT 
    p.product_name,
    c.category_name,
    SUM(s.quantity) AS total_sold,
    SUM(s.quantity * s.unit_price * (1 - s.discount_percent/100)) AS total_revenue,
    AVG(s.discount_percent) AS avg_discount
FROM mysql.sales_db.sales s
JOIN mysql.sales_db.products p ON s.product_id = p.product_id
JOIN mysql.sales_db.categories c ON p.category_id = c.category_id
GROUP BY p.product_name, c.category_name
ORDER BY total_revenue DESC
LIMIT 10
'''

df_top_products = execute_query(query_top_products)
print('🏆 Top-10 продуктов по выручке:')
df_top_products

In [ ]:
# Запрос 6: Продажи по регионам
query_regions = '''
SELECT 
    s.region,
    COUNT(*) AS transactions,
    SUM(s.quantity) AS total_quantity,
    SUM(s.quantity * s.unit_price * (1 - s.discount_percent/100)) AS revenue
FROM mysql.sales_db.sales s
GROUP BY s.region
ORDER BY revenue DESC
'''

df_regions = execute_query(query_regions)
print('🌍 Продажи по регионам:')
df_regions

## 2.3 Cross-Database анализ (PostgreSQL + MySQL)

In [ ]:
# Запрос 7: Сравнение ежемесячной выручки PostgreSQL vs MySQL
query_cross_db = '''
WITH pg_monthly AS (
    SELECT 
        date_trunc('month', order_date) AS month,
        'PostgreSQL Orders' AS source,
        SUM(amount) AS revenue
    FROM postgresql.public.orders
    GROUP BY date_trunc('month', order_date)
),
mysql_monthly AS (
    SELECT 
        date_trunc('month', sale_date) AS month,
        'MySQL Sales' AS source,
        SUM(quantity * unit_price * (1 - discount_percent/100)) AS revenue
    FROM mysql.sales_db.sales
    GROUP BY date_trunc('month', sale_date)
)
SELECT * FROM pg_monthly
UNION ALL
SELECT * FROM mysql_monthly
ORDER BY month, source
'''

df_cross_db = execute_query(query_cross_db)
print('🔄 Сравнение выручки PostgreSQL vs MySQL:')
df_cross_db

## 2.4 Формирование финального DataFrame

In [ ]:
# Финальный агрегированный DataFrame для сохранения в Iceberg
# Объединяем данные из обоих источников по месяцам

query_final = '''
WITH pg_stats AS (
    SELECT 
        date_trunc('month', o.order_date) AS month,
        COUNT(DISTINCT o.order_id) AS pg_orders_count,
        SUM(o.amount) AS pg_orders_revenue,
        COUNT(DISTINCT o.customer_id) AS pg_unique_customers
    FROM postgresql.public.orders o
    GROUP BY date_trunc('month', o.order_date)
),
mysql_stats AS (
    SELECT 
        date_trunc('month', s.sale_date) AS month,
        COUNT(*) AS mysql_transactions,
        SUM(s.quantity) AS mysql_items_sold,
        SUM(s.quantity * s.unit_price * (1 - s.discount_percent/100)) AS mysql_sales_revenue
    FROM mysql.sales_db.sales s
    GROUP BY date_trunc('month', s.sale_date)
)
SELECT 
    COALESCE(pg.month, ms.month) AS report_month,
    COALESCE(pg.pg_orders_count, 0) AS orders_count,
    COALESCE(pg.pg_orders_revenue, 0) AS orders_revenue,
    COALESCE(pg.pg_unique_customers, 0) AS unique_customers,
    COALESCE(ms.mysql_transactions, 0) AS sales_transactions,
    COALESCE(ms.mysql_items_sold, 0) AS items_sold,
    COALESCE(ms.mysql_sales_revenue, 0) AS sales_revenue,
    COALESCE(pg.pg_orders_revenue, 0) + COALESCE(ms.mysql_sales_revenue, 0) AS total_revenue
FROM pg_stats pg
FULL OUTER JOIN mysql_stats ms ON pg.month = ms.month
ORDER BY report_month
'''

df_final = execute_query(query_final)
print('📊 Финальный агрегированный DataFrame:')
df_final

In [ ]:
# Статистика финального DataFrame
print('📈 Статистика финального DataFrame:')
print(f'Количество записей: {len(df_final)}')
print(f'Период: {df_final["report_month"].min()} — {df_final["report_month"].max()}')
print(f'Общая выручка: {df_final["total_revenue"].sum():,.2f} руб.')
print(f'Всего заказов: {df_final["orders_count"].sum()}')
print(f'Всего транзакций продаж: {df_final["sales_transactions"].sum()}')

### ✅ Итоги Уровня 2

Выполнено:
- 7 агрегирующих SQL-запросов через Trino
- Анализ данных из PostgreSQL (заказы, клиенты, сегменты)
- Анализ данных из MySQL (продажи, продукты, категории, регионы)
- Cross-database анализ (сравнение двух источников)
- Сформирован финальный агрегированный DataFrame

---

# Уровень 3: Визуализация и Iceberg

На этом уровне мы:
1. Построим графики на основе агрегированных данных
2. Создадим схему и таблицу в Iceberg
3. Сохраним данные и проверим результат

## 3.1 Визуализация данных

In [ ]:
# График 1: Динамика выручки по месяцам (линейный график)
fig, ax = plt.subplots(figsize=(12, 6))

df_plot = df_final.copy()
df_plot['month_str'] = pd.to_datetime(df_plot['report_month']).dt.strftime('%Y-%m')

ax.plot(df_plot['month_str'], df_plot['orders_revenue']/1000, 
        marker='o', linewidth=2.5, label='Заказы (PostgreSQL)', color=COLORS[0])
ax.plot(df_plot['month_str'], df_plot['sales_revenue']/1000, 
        marker='s', linewidth=2.5, label='Продажи (MySQL)', color=COLORS[1])
ax.plot(df_plot['month_str'], df_plot['total_revenue']/1000, 
        marker='^', linewidth=3, label='Общая выручка', color=COLORS[2], linestyle='--')

ax.set_xlabel('Месяц', fontsize=12, fontweight='bold')
ax.set_ylabel('Выручка (тыс. руб.)', fontsize=12, fontweight='bold')
ax.set_title('Динамика выручки по месяцам\n(PostgreSQL Orders + MySQL Sales)', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# График 2: Top-10 продуктов по выручке (горизонтальная столбчатая диаграмма)
fig, ax = plt.subplots(figsize=(12, 7))

df_top_products_sorted = df_top_products.sort_values('total_revenue', ascending=True)

colors_cat = [COLORS[i % len(COLORS)] for i in range(len(df_top_products_sorted))]
bars = ax.barh(df_top_products_sorted['product_name'], 
               df_top_products_sorted['total_revenue']/1000, 
               color=colors_cat, edgecolor='black', linewidth=0.5)

# Добавляем значения на столбцы
for bar, val in zip(bars, df_top_products_sorted['total_revenue']/1000):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2, 
            f'{val:,.0f}', va='center', fontsize=9)

ax.set_xlabel('Выручка (тыс. руб.)', fontsize=12, fontweight='bold')
ax.set_ylabel('Продукт', fontsize=12, fontweight='bold')
ax.set_title('Top-10 продуктов по выручке\n(данные из MySQL)', 
             fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# График 3: Распределение выручки по сегментам клиентов (pie chart)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart сегментов
ax1 = axes[0]
wedges, texts, autotexts = ax1.pie(
    df_segments['total_revenue'], 
    labels=df_segments['segment'],
    autopct='%1.1f%%',
    colors=COLORS[:len(df_segments)],
    explode=[0.05] * len(df_segments),
    shadow=True,
    startangle=90
)
ax1.set_title('Распределение выручки\nпо сегментам клиентов', 
              fontsize=13, fontweight='bold')

# Pie chart регионов
ax2 = axes[1]
wedges2, texts2, autotexts2 = ax2.pie(
    df_regions['revenue'], 
    labels=df_regions['region'],
    autopct='%1.1f%%',
    colors=COLORS[:len(df_regions)],
    explode=[0.03] * len(df_regions),
    shadow=True,
    startangle=90
)
ax2.set_title('Распределение выручки\nпо регионам', 
              fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# График 4: Heatmap продаж по категориям и месяцам
fig, ax = plt.subplots(figsize=(12, 6))

# Подготовка данных для heatmap
df_heatmap = df_sales_category.copy()
df_heatmap['month_str'] = pd.to_datetime(df_heatmap['month']).dt.strftime('%Y-%m')
pivot_table = df_heatmap.pivot_table(
    values='revenue', 
    index='category_name', 
    columns='month_str', 
    aggfunc='sum',
    fill_value=0
)

sns.heatmap(pivot_table/1000, annot=True, fmt='.0f', cmap='YlOrRd', 
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Выручка (тыс. руб.)'})

ax.set_xlabel('Месяц', fontsize=12, fontweight='bold')
ax.set_ylabel('Категория', fontsize=12, fontweight='bold')
ax.set_title('Тепловая карта продаж по категориям и месяцам', 
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3.2 Сохранение данных в Iceberg

In [ ]:
# Создание схемы в Iceberg (если не существует)
conn = get_trino_connection('iceberg')
cursor = conn.cursor()

create_schema_sql = '''
CREATE SCHEMA IF NOT EXISTS iceberg.analytics
WITH (location = 's3a://warehouse/analytics/')
'''

cursor.execute(create_schema_sql)
print('✅ Схема iceberg.analytics создана (или уже существует)')

In [ ]:
# Проверка созданной схемы
iceberg_schemas = execute_query('SHOW SCHEMAS FROM iceberg')
print('📂 Схемы в каталоге Iceberg:')
iceberg_schemas

In [ ]:
# Удаление таблицы если существует (для повторного запуска)
conn = get_trino_connection('iceberg')
cursor = conn.cursor()

try:
    cursor.execute('DROP TABLE IF EXISTS iceberg.analytics.monthly_report')
    print('🗑️ Старая таблица удалена (если существовала)')
except Exception as e:
    print(f'Примечание: {e}')

In [ ]:
# Создание Iceberg таблицы с партиционированием по месяцам
create_table_sql = '''
CREATE TABLE iceberg.analytics.monthly_report (
    report_month DATE,
    orders_count INTEGER,
    orders_revenue DECIMAL(15, 2),
    unique_customers INTEGER,
    sales_transactions INTEGER,
    items_sold INTEGER,
    sales_revenue DECIMAL(15, 2),
    total_revenue DECIMAL(15, 2)
)
WITH (
    format = 'PARQUET',
    partitioning = ARRAY['month(report_month)']
)
'''

conn = get_trino_connection('iceberg')
cursor = conn.cursor()
cursor.execute(create_table_sql)
print('✅ Таблица iceberg.analytics.monthly_report создана с партиционированием по месяцам')

In [ ]:
# Сохранение агрегированных данных в Iceberg
insert_sql = '''
INSERT INTO iceberg.analytics.monthly_report
WITH pg_stats AS (
    SELECT 
        CAST(date_trunc('month', o.order_date) AS DATE) AS month,
        COUNT(DISTINCT o.order_id) AS pg_orders_count,
        SUM(o.amount) AS pg_orders_revenue,
        COUNT(DISTINCT o.customer_id) AS pg_unique_customers
    FROM postgresql.public.orders o
    GROUP BY CAST(date_trunc('month', o.order_date) AS DATE)
),
mysql_stats AS (
    SELECT 
        CAST(date_trunc('month', s.sale_date) AS DATE) AS month,
        COUNT(*) AS mysql_transactions,
        SUM(s.quantity) AS mysql_items_sold,
        SUM(s.quantity * s.unit_price * (1 - s.discount_percent/100)) AS mysql_sales_revenue
    FROM mysql.sales_db.sales s
    GROUP BY CAST(date_trunc('month', s.sale_date) AS DATE)
)
SELECT 
    COALESCE(pg.month, ms.month) AS report_month,
    CAST(COALESCE(pg.pg_orders_count, 0) AS INTEGER) AS orders_count,
    CAST(COALESCE(pg.pg_orders_revenue, 0) AS DECIMAL(15,2)) AS orders_revenue,
    CAST(COALESCE(pg.pg_unique_customers, 0) AS INTEGER) AS unique_customers,
    CAST(COALESCE(ms.mysql_transactions, 0) AS INTEGER) AS sales_transactions,
    CAST(COALESCE(ms.mysql_items_sold, 0) AS INTEGER) AS items_sold,
    CAST(COALESCE(ms.mysql_sales_revenue, 0) AS DECIMAL(15,2)) AS sales_revenue,
    CAST(COALESCE(pg.pg_orders_revenue, 0) + COALESCE(ms.mysql_sales_revenue, 0) AS DECIMAL(15,2)) AS total_revenue
FROM pg_stats pg
FULL OUTER JOIN mysql_stats ms ON pg.month = ms.month
'''

conn = get_trino_connection('iceberg')
cursor = conn.cursor()
cursor.execute(insert_sql)
print('✅ Данные успешно сохранены в iceberg.analytics.monthly_report')

In [ ]:
# Проверочный SQL-запрос к Iceberg таблице
verify_df = execute_query('SELECT * FROM iceberg.analytics.monthly_report ORDER BY report_month')
print('🔍 Проверка данных в Iceberg таблице:')
verify_df

In [ ]:
# Проверка метаданных таблицы
table_info = execute_query('DESCRIBE iceberg.analytics.monthly_report')
print('📋 Структура таблицы в Iceberg:')
table_info

## 3.3 Дополнительно: работа со снапшотами Iceberg

In [ ]:
# Просмотр истории снапшотов (версий) таблицы
snapshots_query = '''
SELECT 
    snapshot_id,
    parent_id,
    operation,
    manifest_list,
    committed_at
FROM iceberg.analytics."monthly_report$snapshots"
ORDER BY committed_at DESC
'''

try:
    snapshots_df = execute_query(snapshots_query)
    print('📸 История снапшотов Iceberg таблицы:')
    display(snapshots_df)
except Exception as e:
    print(f'Примечание: Снапшоты недоступны ({e})')

In [ ]:
# Просмотр партиций таблицы
partitions_query = '''
SELECT 
    partition,
    record_count,
    file_count
FROM iceberg.analytics."monthly_report$partitions"
'''

try:
    partitions_df = execute_query(partitions_query)
    print('📁 Партиции Iceberg таблицы:')
    display(partitions_df)
except Exception as e:
    print(f'Примечание: Информация о партициях недоступна ({e})')

In [ ]:
# Агрегирующий запрос к Iceberg для проверки
agg_query = '''
SELECT 
    COUNT(*) AS total_months,
    SUM(orders_count) AS total_orders,
    SUM(sales_transactions) AS total_sales,
    SUM(total_revenue) AS grand_total_revenue,
    AVG(total_revenue) AS avg_monthly_revenue
FROM iceberg.analytics.monthly_report
'''

agg_df = execute_query(agg_query)
print('📊 Сводная статистика из Iceberg:')
agg_df

### ✅ Итоги Уровня 3

Выполнено:
- Построено 4 графика с оформлением (подписи осей, заголовки, легенды):
  - Линейный график динамики выручки
  - Столбчатая диаграмма Top-10 продуктов
  - Круговые диаграммы распределения по сегментам и регионам
  - Тепловая карта продаж по категориям
- Создана схема `iceberg.analytics` в каталоге Iceberg
- Создана таблица `monthly_report` с партиционированием по месяцам
- Данные сохранены в Iceberg
- Выполнены проверочные запросы

---

# Заключение

## Выполненные задачи

| Уровень | Задача | Статус |
|---------|--------|--------|
| 1 | Подключение к Trino | ✅ |
| 1 | Проверка каталога PostgreSQL | ✅ |
| 1 | Проверка каталога MySQL | ✅ |
| 2 | Агрегирующие SQL-запросы (7 шт.) | ✅ |
| 2 | Загрузка в pandas DataFrame | ✅ |
| 2 | Финальный агрегированный DataFrame | ✅ |
| 3 | Визуализация (4 графика) | ✅ |
| 3 | Создание схемы Iceberg | ✅ |
| 3 | Создание таблицы с партиционированием | ✅ |
| 3 | Сохранение данных в Iceberg | ✅ |
| 3 | Проверочные SQL-запросы | ✅ |

## Дополнительные возможности (бонус)

- Cross-database анализ (объединение данных PostgreSQL и MySQL)
- Партиционирование таблицы Iceberg по месяцам
- Работа со снапшотами Iceberg
- Расширенная визуализация (тепловая карта, множественные графики)

---

**Автор:** Литвинов Никита Антонович, группа МИНДА 241